In [10]:
import gzip, io
import numpy as np
import pandas as pd

SERIES_MATRIX = r"C:\Users\arz_2\Basira_Motassim_Group_A\validation_data\GSE62254_series_matrix\GSE62254_series_matrix.txt"   
meta, table_lines, in_table = {}, [], False
f = open(SERIES_MATRIX, "rt", encoding="utf-8", errors="replace")
for line in f:
    line = line.rstrip("\n")
    if line.startswith("!series_matrix_table_begin"):
        in_table = True
        continue
    if line.startswith("!series_matrix_table_end"):
        in_table = False
        continue
    if in_table:
        table_lines.append(line)
    elif line.startswith("!Sample_"):
        parts = line.split("\t")
        meta.setdefault(parts[0], []).append([p.strip('"') for p in parts[1:]])

# expression: probes x samples
expr = pd.read_csv(io.StringIO("\n".join(table_lines)), sep="\t", index_col=0)
expr.index = expr.index.astype(str).str.strip('"')
expr.columns = [c.strip('"') for c in expr.columns]

# clinical: one row per sample
gsm = meta["!Sample_geo_accession"][0]
clin = pd.DataFrame({"GSM": gsm, "title": meta["!Sample_title"][0]}, index=gsm)

for row in meta.get("!Sample_characteristics_ch1", []):
    pairs = [v.split(": ", 1) if ": " in v else ["", v] for v in row]
    keys = [p[0] for p in pairs if p[0]]
    if not keys:
        continue
    key = pd.Series(keys).mode()[0]
    clin[key] = [p[1] if p[0] == key else np.nan for p in pairs]

print("Probes x samples:", expr.shape)
print("Value range:", float(np.nanmin(expr.values)), float(np.nanmax(expr.values)))
print("Median of sample medians:", float(expr.median(axis=0).median()))
print("Any NA:", bool(expr.isna().any().any()))
print("Platform:", meta["!Sample_platform_id"][0][0], "\n")
print("Clinical shape:", clin.shape)
print(list(clin.columns), "\n")
print(clin.head())

for c in clin.columns[2:]:
    print("\n--", c)
    print(clin[c].value_counts(dropna=False).head(6).to_string())

#expr.to_pickle("GSE62254_expr1.pkl")
#clin.to_csv("GSE62254_clinical(1).csv")

Probes x samples: (54675, 300)
Value range: 0.572611153 4.412371159
Median of sample medians: 1.4759190675
Any NA: False
Platform: GPL570 

Clinical shape: (300, 4)
['GSM', 'title', 'tissue', 'patient'] 

                   GSM title         tissue patient
GSM1523727  GSM1523727  T107  Gastric tumor     107
GSM1523728  GSM1523728  T108  Gastric tumor     108
GSM1523729  GSM1523729  T109  Gastric tumor     109
GSM1523744  GSM1523744  T160  Gastric tumor     160
GSM1523745  GSM1523745  T164  Gastric tumor     164

-- tissue
tissue
Gastric tumor    300

-- patient
patient
107    1
108    1
109    1
160    1
164    1
165    1


In [11]:
import gzip

SOFT = r"C:\Users\arz_2\Basira_Motassim_Group_A\validation_data\GSE62254_family\GSE62254_family.soft"   # edit this

stream = open(SOFT, "rt", encoding="utf-8", errors="replace")

n_samples = 0
with stream as f:
    for line in f:
        if line.startswith("^SAMPLE"):
            n_samples += 1
            if n_samples == 2:
                break
            print("\n", line.strip())
            continue
        if n_samples == 1:
            if line.startswith("!sample_table_begin"):
                continue
            if line.startswith("!Sample_"):
                print(line.strip()[:250])



 ^SAMPLE = GSM1523727
!Sample_title = T107
!Sample_geo_accession = GSM1523727
!Sample_status = Public on Apr 17 2015
!Sample_submission_date = Oct 10 2014
!Sample_last_update_date = Apr 17 2015
!Sample_type = RNA
!Sample_channel_count = 1
!Sample_source_name_ch1 = Tumor from patient 107
!Sample_organism_ch1 = Homo sapiens
!Sample_taxid_ch1 = 9606
!Sample_characteristics_ch1 = tissue: Gastric tumor
!Sample_characteristics_ch1 = patient: 107
!Sample_molecule_ch1 = total RNA
!Sample_extract_protocol_ch1 = Samples are processed in parallel in 96-well plates to minimize potential variation. Reaction purification is achieved using magnetic binding beads for cDNA and Qiagen RNeasy kits for cRNA purification.
!Sample_label_ch1 = biotin
!Sample_label_protocol_ch1 = standard Affymetrix protocol
!Sample_hyb_protocol = GeneChip microarrays are loaded with the fragmented target sample/hybridization buffer mix using standard manual techniques. Arrays are hybridized for 18 hours at 45oC with vigorou

In [12]:
import io, pandas as pd, numpy as np

SOFT = r"C:\Users\arz_2\Basira_Motassim_Group_A\validation_data\GSE62254_family\GSE62254_family.soft"   # same file as before

lines, in_tab = [], False
with open(SOFT, "rt", encoding="utf-8", errors="replace") as f:
    for line in f:
        if line.startswith("!platform_table_begin"):
            in_tab = True
            continue
        if line.startswith("!platform_table_end"):
            break
        if in_tab:
            lines.append(line)
        elif line.startswith("^SAMPLE"):
            break

if not lines:
    print("No platform table in this file. Download the annotation table from the GPL570 page instead.")
else:
    annot = pd.read_csv(io.StringIO("".join(lines)), sep="\t", dtype=str)
    print(annot.shape)
    print(list(annot.columns)[:12])
    annot.to_csv("GPL570_annotation.csv", index=False)

    # scale check: which probes are highest, and what would they be on log2?
    top = expr.mean(axis=1).sort_values(ascending=False).head(8)
    sym = annot.set_index("ID")["Gene Symbol"] if "Gene Symbol" in annot.columns else None
    out = pd.DataFrame({"mean_in_matrix": top.round(3),
                        "if_log10_to_log2": (top * np.log2(10)).round(2)})
    if sym is not None:
        out["gene"] = sym.reindex(out.index).values
    print(out)

(54675, 16)
['ID', 'GB_ACC', 'SPOT_ID', 'Species Scientific Name', 'Annotation Date', 'Sequence Type', 'Sequence Source', 'Target Description', 'Representative Public ID', 'Gene Title', 'Gene Symbol', 'ENTREZ_GENE_ID']
                 mean_in_matrix  if_log10_to_log2                   gene
ID_REF                                                                  
1553588_at                4.281             14.22        ND3 /// SH3KBP1
201492_s_at               4.246             14.11                  RPL41
204892_x_at               4.223             14.03                 EEF1A1
1553570_x_at              4.214             14.00                   COX2
212284_x_at               4.207             13.98  LOC101928826 /// TPT1
AFFX-hum_alu_at           4.203             13.96                    NaN
207783_x_at               4.202             13.96                  HUWE1
212869_x_at               4.200             13.95  LOC101928826 /// TPT1


In [13]:
import gzip, numpy as np, pandas as pd

M66229 = r"C:\Users\arz_2\Basira_Motassim_Group_A\validation_data\GSE62254_series_matrix.txt.gz"   # edit this

def read_clin(path):
    opn = gzip.open if path.endswith(".gz") else open
    meta = {}
    with opn(path, "rt", encoding="utf-8", errors="replace") as f:
        for line in f:
            if line.startswith("!series_matrix_table_begin"):
                break
            if line.startswith("!Sample_"):
                parts = line.rstrip("\n").split("\t")
                meta.setdefault(parts[0], []).append([p.strip('"') for p in parts[1:]])
    gsm = meta["!Sample_geo_accession"][0]
    df = pd.DataFrame({"GSM": gsm, "title": meta["!Sample_title"][0]}, index=gsm)
    for row in meta.get("!Sample_characteristics_ch1", []):
        pairs = [v.split(": ", 1) if ": " in v else ["", v] for v in row]
        keys = [p[0] for p in pairs if p[0]]
        if not keys:
            continue
        key = pd.Series(keys).mode()[0]
        df[key] = [p[1] if p[0] == key else np.nan for p in pairs]
    return df

clin2 = read_clin(M66229)
print(clin2.shape)
print(list(clin2.columns))
print("GSMs shared with our 300:", len(set(clin2.index) & set(clin.index)))

for c in clin2.columns[2:]:
    print("\n--", c)
    print(clin2[c].value_counts(dropna=False).head(8).to_string())

(300, 4)
['GSM', 'title', 'tissue', 'patient']
GSMs shared with our 300: 300

-- tissue
tissue
Gastric tumor    300

-- patient
patient
107    1
108    1
109    1
160    1
164    1
165    1
168    1
169    1


In [14]:
import numpy as np, pandas as pd

# scale: matrix appears to be log10(RMA); convert to log2 (inference, see check below)
expr_log2 = expr * np.log2(10)
print("log2-scale range:", round(float(expr_log2.values.min()), 2), "to", round(float(expr_log2.values.max()), 2))

annot = pd.read_csv("GPL570_annotation.csv", dtype=str).set_index("ID")
sym = annot["Gene Symbol"].reindex(expr_log2.index)

is_multi = sym.str.contains("///", na=False)
is_none  = sym.isna()
is_affx  = pd.Series(expr_log2.index.str.startswith("AFFX"), index=expr_log2.index)
keep = ~(is_multi | is_none | is_affx)
print(f"probes {len(sym)} | no symbol {is_none.sum()} | multi-gene {is_multi.sum()} | control {is_affx.sum()} | kept {keep.sum()}")

# rule: one probe per gene, the one with the highest mean expression
e = expr_log2.loc[keep].copy()
m = e.mean(axis=1).values
e["gene"] = sym[keep].str.strip().values
e = e.assign(_m=m).sort_values("_m", ascending=False).drop_duplicates("gene")
genes = e.set_index("gene").drop(columns="_m")
print("genes x samples:", genes.shape)

CAD = ["CDH1","CDH2","CDH3","CDH11","CDH13","CDH17","CDHR2","CDHR5",
       "CLSTN1","DSC2","DSG2","FAT1","PCDH1","PCDH17","PCDH18","PCDHGA10"]
need = ["TP53", "GZMA", "PRF1"] + CAD
print({g: (g in genes.index) for g in need})

genes.to_csv("ACRG_log2_genes.tsv", sep="\t")

log2-scale range: 1.9 to 14.66
probes 54675 | no symbol 8893 | multi-gene 2796 | control 62 | kept 42976
genes x samples: (21655, 300)
{'TP53': True, 'GZMA': True, 'PRF1': True, 'CDH1': True, 'CDH2': True, 'CDH3': True, 'CDH11': True, 'CDH13': True, 'CDH17': True, 'CDHR2': True, 'CDHR5': True, 'CLSTN1': True, 'DSC2': True, 'DSG2': True, 'FAT1': True, 'PCDH1': True, 'PCDH17': True, 'PCDH18': True, 'PCDHGA10': True}


In [15]:
import subprocess, shutil
from pathlib import Path
import numpy as np, pandas as pd

output_dir = Path("ACRG_Analysis")
output_dir.mkdir(exist_ok=True)

genes.index.name = "GeneSymbol"
expr_file = output_dir / "ACRG_log2_genes.tsv"
genes.to_csv(expr_file, sep="\t")

# cytolytic activity: mean of log2 GZMA and PRF1 (= geometric mean on the linear scale)
cyt = (genes.loc["GZMA"] + genes.loc["PRF1"]) / 2
cyt_df = pd.DataFrame({"GSM": cyt.index, "CYT_Score": cyt.values})

r_scripts = list(Path(r"C:\Program Files\R").glob("R-*/bin/Rscript.exe"))
rscript = str(r_scripts[-1]) if r_scripts else shutil.which("Rscript")
print("Rscript:", rscript)
print(cyt_df["CYT_Score"].describe())

Rscript: C:\Program Files\R\R-4.6.2\bin\Rscript.exe
count    300.000000
mean       7.639819
std        1.019916
min        4.468119
25%        6.956224
50%        7.598813
75%        8.260730
max       10.828270
Name: CYT_Score, dtype: float64


In [21]:
! "C:\Program Files\R\R-4.6.2\bin\Rscript.exe" -e "lib <- file.path(path.expand('~'), 'R_libs'); dir.create(lib, showWarnings=FALSE); .libPaths(c(lib, .libPaths())); install.packages('remotes', repos='https://cloud.r-project.org')"

! "C:\Program Files\R\R-4.6.2\bin\Rscript.exe" -e "lib <- file.path(path.expand('~'), 'R_libs'); .libPaths(c(lib, .libPaths())); install.packages('estimate', repos='http://R-Forge.R-project.org', type='source', dependencies=TRUE); remotes::install_github('ebecht/MCPcounter', ref='master', subdir='Source', upgrade='never')"

package 'remotes' successfully unpacked and MD5 sums checked

The downloaded binary packages are in
	C:\Users\arz_2\AppData\Local\Temp\RtmpEVrOOw\downloaded_packages


Installing package into 'C:/Users/arz_2/Documents/R_libs'
(as 'lib' is unspecified)
trying URL 'https://cloud.r-project.org/bin/windows/contrib/4.6/remotes_2.5.0.zip'
Content type 'application/zip' length 439980 bytes (429 KB)
downloaded 429 KB



package 'curl' successfully unpacked and MD5 sums checked

The downloaded binary packages are in
	C:\Users\arz_2\AppData\Local\Temp\RtmpOuS3Q6\downloaded_packages
* checking for file 'C:\Users\arz_2\AppData\Local\Temp\RtmpOuS3Q6\remotesf982c64737\ebecht-MCPcounter-b6eac73\Source/DESCRIPTION' ... OK
* preparing 'MCPcounter':
* checking DESCRIPTION meta-information ... OK
* checking for LF line-endings in source and make files and shell scripts
* checking for empty or unneeded directories
* building 'MCPcounter_1.2.0.tar.gz'


Installing package into 'C:/Users/arz_2/Documents/R_libs'
(as 'lib' is unspecified)
trying URL 'http://R-Forge.R-project.org/src/contrib/estimate_1.0.13.tar.gz'
Content type 'application/x-gzip' length 3845282 bytes (3.7 MB)
downloaded 3.7 MB

* installing *source* package 'estimate' ...
** this is package 'estimate' version '1.0.13'
** using staged installation
** R
** data
*** moving datasets to lazyload DB
** inst
** byte-compile and prepare package for lazy loading
** help
*** installing help indices
** building package indices
** installing vignettes
** testing if installed package can be loaded from temporary location
** testing if installed package can be loaded from final location
** testing if installed package keeps a record of temporary installation path
* DONE (estimate)

The downloaded source packages are in
	'C:\Users\arz_2\AppData\Local\Temp\RtmpOuS3Q6\downloaded_packages'
Using GitHub PAT from the git credential store.
Installing 1 packages: curl
Installing package into

In [22]:
r_code = r'''
.libPaths(c(file.path(path.expand("~"), "R_libs"), .libPaths()))
library(estimate); library(MCPcounter)
args <- commandArgs(trailingOnly = TRUE)
inp <- args[1]; outdir <- args[2]

# ESTIMATE
filt <- file.path(outdir, "ESTIMATE_Filtered.gct")
res  <- file.path(outdir, "ESTIMATE_Results.gct")
filterCommonGenes(input.f = inp, output.f = filt, id = "GeneSymbol")
estimateScore(input.ds = filt, output.ds = res, platform = "affymetrix")
sc  <- read.delim(res, skip = 2, row.names = 1, check.names = FALSE)
ids <- setdiff(colnames(sc), "Description")
write.csv(data.frame(GSM = ids,
                     ESTIMATE_Immune_Score  = as.numeric(sc["ImmuneScore", ids]),
                     ESTIMATE_Stromal_Score = as.numeric(sc["StromalScore", ids])),
          file.path(outdir, "ACRG_ESTIMATE.csv"), row.names = FALSE)

# MCP-counter, all populations
expr <- as.matrix(read.delim(inp, row.names = 1, check.names = FALSE))
mcp  <- MCPcounter.estimate(expr, featuresType = "HUGO_symbols")
write.csv(data.frame(GSM = colnames(mcp), t(mcp), check.names = FALSE),
          file.path(outdir, "ACRG_MCPcounter_all.csv"), row.names = FALSE)
cat("done\n")
'''
script = output_dir / "acrg_scores.R"
script.write_text(r_code, encoding="utf-8")
res = subprocess.run([rscript, str(script.resolve()),
                      str(expr_file.resolve()), str(output_dir.resolve())],
                     capture_output=True, text=True, errors="replace")
print("Return code:", res.returncode)
print(res.stdout[-2500:])
print(res.stderr[-2500:])

Return code: 0
[1] "Merged dataset includes 10035 genes (377 mismatched)."
[1] "1 gene set: StromalSignature  overlap= 136"
[1] "2 gene set: ImmuneSignature  overlap= 137"
done

Loading required package: curl
Using libcurl 8.14.1-DEV with Schannel



In [23]:
est = pd.read_csv(output_dir / "ACRG_ESTIMATE.csv")
mcp = pd.read_csv(output_dir / "ACRG_MCPcounter_all.csv")
print("MCP populations:", list(mcp.columns[1:]))

scores = (est.merge(mcp[["GSM", "CD8 T cells"]].rename(columns={"CD8 T cells": "CD8_Score"}),
                    on="GSM", validate="one_to_one")
             .merge(cyt_df, on="GSM", validate="one_to_one"))

cols = ["ESTIMATE_Immune_Score", "CD8_Score", "CYT_Score"]
assert len(scores) == 300 and scores["GSM"].is_unique
assert np.isfinite(scores[cols].to_numpy()).all()
print(scores[cols].describe().round(2))

scores.to_csv(output_dir / "ACRG_Immune_Scores.csv", index=False)

MCP populations: ['T cells', 'CD8 T cells', 'Cytotoxic lymphocytes', 'B lineage', 'NK cells', 'Monocytic lineage', 'Myeloid dendritic cells', 'Neutrophils', 'Endothelial cells', 'Fibroblasts']
       ESTIMATE_Immune_Score  CD8_Score  CYT_Score
count                 300.00     300.00     300.00
mean                 1207.22       3.52       7.64
std                   725.04       0.20       1.02
min                  -726.09       2.99       4.47
25%                   716.49       3.38       6.96
50%                  1227.63       3.51       7.60
75%                  1703.32       3.65       8.26
max                  3203.21       4.18      10.83
